[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/proyectos-integradores/02-chatbot-soporte.ipynb)

# Proyecto 2: Chatbot de soporte con memoria

Explora el sistema de soporte: base de conocimiento, memoria de sesión,
enrutamiento Haiku→Sonnet y escalado a humano.

In [ ]:
import anthropic
import chromadb
import json
import uuid
from datetime import datetime

client = anthropic.Anthropic()
print("Cliente listo")

## 1. Base de conocimiento en ChromaDB

In [ ]:
ARTICULOS = [
    {"id": "kb_001", "titulo": "Resetear contraseña",
     "contenido": "Para resetear tu contraseña ve a Configuración → Seguridad → Cambiar contraseña. El enlace expira en 24 horas."},
    {"id": "kb_002", "titulo": "Planes y precios",
     "contenido": "Starter: 9€/mes (1 usuario). Pro: 29€/mes (5 usuarios). Business: 99€/mes (ilimitados). Descuento del 20% en pago anual."},
    {"id": "kb_003", "titulo": "API e integraciones",
     "contenido": "API REST disponible en planes Pro y Business. Rate limit: 100 req/min en Pro, 1000 en Business. Auth: Bearer token."},
    {"id": "kb_004", "titulo": "Exportar datos",
     "contenido": "Exporta en CSV o JSON desde Configuración → Datos → Exportar. Recibirás email con enlace de descarga."},
    {"id": "kb_005", "titulo": "Cancelar suscripción",
     "contenido": "Cancela en Configuración → Facturación → Cancelar. Conservas el acceso hasta fin del periodo pagado. Sin penalización."},
    {"id": "kb_006", "titulo": "Soporte técnico",
     "contenido": "Soporte por email en soporte@tuproducto.com. Tiempo de respuesta: 4h en Business, 24h en Pro, 72h en Starter."},
]

chroma = chromadb.Client()
col = chroma.get_or_create_collection("kb")

for art in ARTICULOS:
    col.add(
        ids=[art["id"]],
        documents=[f"{art['titulo']}\n{art['contenido']}"],
        metadatas=[{"titulo": art["titulo"]}]
    )

def buscar_kb(pregunta: str, n: int = 3) -> list[dict]:
    res = col.query(query_texts=[pregunta], n_results=min(n, col.count()))
    return [
        {"contenido": doc, "titulo": meta["titulo"], "relevancia": round(1-dist, 3)}
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]

# Probar búsqueda
resultados = buscar_kb("¿Cómo cancelo mi cuenta?")
print(f"Resultados para 'cancelar cuenta':")
for r in resultados:
    print(f"  [{r['relevancia']:.2f}] {r['titulo']}")

## 2. Gestión de sesión y memoria

In [ ]:
# Sesiones en memoria (equivalente al SQLite del proyecto real)
SESIONES: dict[str, list[dict]] = {}

def guardar_mensaje(session_id: str, rol: str, contenido: str):
    if session_id not in SESIONES:
        SESIONES[session_id] = []
    SESIONES[session_id].append({"role": rol, "content": contenido})

def obtener_historial(session_id: str, max_msg: int = 8) -> list[dict]:
    return SESIONES.get(session_id, [])[-max_msg:]

print("Sistema de sesiones listo")

## 3. Motor de respuesta: Haiku → Sonnet → Escalado

In [ ]:
SYSTEM_SOPORTE = """Eres el asistente de soporte de TuProducto SaaS.
Usa SOLO la información de la base de conocimiento proporcionada.
Si no tienes la información, di exactamente: 'NECESITO_ESCALAR'
Respuestas cortas (máx 3 párrafos). Tono: amigable y profesional."""

METRICAS = {"haiku": 0, "sonnet": 0, "escalados": 0, "total": 0}

def responder(session_id: str, pregunta: str) -> dict:
    METRICAS["total"] += 1
    
    # Recuperar contexto
    historial = obtener_historial(session_id)
    contexto_kb = buscar_kb(pregunta)
    contexto_texto = "\n\n".join([
        f"[{r['titulo']}]\n{r['contenido']}"
        for r in contexto_kb if r["relevancia"] > 0.2
    ]) or "Sin artículos relevantes."
    
    # Mensaje enriquecido con KB
    mensajes = historial + [{
        "role": "user",
        "content": f"BASE DE CONOCIMIENTO:\n{contexto_texto}\n\nPREGUNTA: {pregunta}"
    }]
    
    # Intento 1: Haiku (rápido y barato)
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        system=SYSTEM_SOPORTE,
        messages=mensajes
    )
    respuesta = resp.content[0].text
    modelo = "haiku"
    
    # ¿Necesita escalar? Reintento con Sonnet
    if "NECESITO_ESCALAR" in respuesta:
        resp2 = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            system=SYSTEM_SOPORTE,
            messages=mensajes
        )
        respuesta = resp2.content[0].text
        modelo = "sonnet"
    
    # ¿Sigue sin saber? → Escalar a humano
    necesita_escalado = "NECESITO_ESCALAR" in respuesta
    if necesita_escalado:
        respuesta = "No tengo suficiente información para responder esta pregunta. Te conectaré con un agente humano en menos de 2 horas."
        METRICAS["escalados"] += 1
    else:
        METRICAS[modelo] += 1
    
    # Guardar en sesión
    guardar_mensaje(session_id, "user", pregunta)
    guardar_mensaje(session_id, "assistant", respuesta)
    
    return {
        "respuesta": respuesta,
        "modelo": "escalado" if necesita_escalado else modelo,
        "necesita_escalado": necesita_escalado,
        "articulos_usados": [r["titulo"] for r in contexto_kb if r["relevancia"] > 0.2]
    }

print("Motor de soporte listo")

In [ ]:
# Simular una sesión de soporte completa
session_id = str(uuid.uuid4())[:8]
print(f"Nueva sesión: {session_id}\n")

PREGUNTAS = [
    "¿Cuánto cuesta el plan Pro?",
    "¿Y si quiero 10 usuarios?",  # Recuerda el contexto de la pregunta anterior
    "¿Cómo cancelo si no me convence?",
    "¿Puedo integrar con mi ERP Navision?",  # Probablemente escalará
]

for pregunta in PREGUNTAS:
    print(f"👤 {pregunta}")
    resultado = responder(session_id, pregunta)
    icono = "🤖" if not resultado["necesita_escalado"] else "👨‍💼"
    print(f"{icono} [{resultado['modelo']}] {resultado['respuesta']}")
    if resultado["articulos_usados"]:
        print(f"   📚 Fuentes: {', '.join(resultado['articulos_usados'])}")
    print()

In [ ]:
# Ver estadísticas de la sesión
print("ESTADÍSTICAS DE USO")
print("=" * 40)
total = METRICAS["total"]
print(f"Total preguntas:    {total}")
print(f"Respondidas Haiku:  {METRICAS['haiku']} ({METRICAS['haiku']/total*100:.0f}%)")
print(f"Respondidas Sonnet: {METRICAS['sonnet']} ({METRICAS['sonnet']/total*100:.0f}%)")
print(f"Escaladas a humano: {METRICAS['escalados']} ({METRICAS['escalados']/total*100:.0f}%)")

# Historial de la sesión
print(f"\nMensajes en sesión {session_id}: {len(SESIONES.get(session_id, []))}")

## 4. Coste estimado de producción

In [ ]:
# Proyección: 500 usuarios × 5 preguntas/día
preguntas_dia = 500 * 5
pct_haiku = 0.75  # 75% resueltas por Haiku
pct_sonnet = 0.15  # 15% necesitan Sonnet
pct_escalado = 0.10  # 10% escalan a humano (no tienen coste de API)

# Tokens promedio por pregunta
tok_in_haiku = 800  # pregunta + historial + KB
tok_out_haiku = 200
tok_in_sonnet = 1200
tok_out_sonnet = 400

coste_haiku = (tok_in_haiku * 0.80 + tok_out_haiku * 4.00) / 1_000_000 * preguntas_dia * pct_haiku
coste_sonnet = (tok_in_sonnet * 3.00 + tok_out_sonnet * 15.00) / 1_000_000 * preguntas_dia * pct_sonnet

print(f"PROYECCIÓN COSTES — 500 usuarios × 5 preguntas/día")
print(f"{'─'*45}")
print(f"Preguntas/día total:  {preguntas_dia:,}")
print(f"Haiku ({pct_haiku*100:.0f}%):          ${coste_haiku:.2f}/día")
print(f"Sonnet ({pct_sonnet*100:.0f}%):         ${coste_sonnet:.2f}/día")
print(f"Total/día:            ${coste_haiku + coste_sonnet:.2f}")
print(f"Total/mes:            ${(coste_haiku + coste_sonnet) * 30:.2f}")
print(f"\nCoste por usuario/mes: ${(coste_haiku + coste_sonnet) * 30 / 500:.3f}")